# Notebook 05 â€” Entity Matching Baseline

Tasks:
- exact match
- normalized match
- fuzzy baseline
- blocking
- candidate recall

In [ ]:
import pandas as pd
import numpy as np

OUTPUT_DIR = "D:/data viewer/data/processed"

## Stage 1: Deterministic Baseline

In [ ]:
def normalize_text(s):
    if pd.isna(s):
        return ""
    return str(s).strip().lower()

def jaro_winkler_similarity(s1, s2):
    s1, s2 = normalize_text(s1), normalize_text(s2)
    if not s1 or not s2:
        return 0.0
    if s1 == s2:
        return 1.0
    len1, len2 = len(s1), len(s2)
    match_distance = max(len1, len2) // 2 - 1
    s1_matches = [False] * len1
    s2_matches = [False] * len2
    matches = 0
    transpositions = 0
    for i in range(len1):
        start = max(0, i - match_distance)
        end = min(i + match_distance + 1, len2)
        for j in range(start, end):
            if s2_matches[j]:
                continue
            if s1[i] != s2[j]:
                continue
            s1_matches[i] = True
            s2_matches[j] = True
            matches += 1
            break
    if matches == 0:
        return 0.0
    k = 0
    for i in range(len1):
        if not s1_matches[i]:
            continue
        while not s2_matches[k]:
            k += 1
        if s1[i] != s2[k]:
            transpositions += 1
        k += 1
    transpositions //= 2
    jaro = (matches / len1 + matches / len2 + (matches - transpositions) / matches) / 3
    prefix = 0
    for i in range(min(4, len1, len2)):
        if s1[i] == s2[i]:
            prefix += 1
        else:
            break
    return jaro + prefix * 0.1 * (1 - jaro)

## Stage 2: Blocking

In [ ]:
pairs = pd.read_csv(f"{OUTPUT_DIR}/entity_pairs.csv")
print(f"Total pairs: {len(pairs)}")

# Generate blocking keys
def get_blocking_key(row):
    state_a = str(row.get('state_a', '')).lower()
    state_b = str(row.get('state_b', '')).lower()
    if state_a == state_b and state_a != '':
        return f"state_{state_a}"
    return "no_block"

# For simplicity, we'll use a reduced pair set for blocking demo
# In production, this would be applied to all possible pairs
blocking_demo = pairs.head(1000).copy()
blocking_demo['block_key'] = blocking_demo.apply(get_blocking_key, axis=1)

candidate_count_before = len(blocking_demo)
blocked = blocking_demo[blocking_demo['block_key'] != 'no_block']
candidate_count_after = len(blocked)

print(f"Candidate count before blocking: {candidate_count_before}")
print(f"Candidate count after blocking: {candidate_count_after}")
print(f"Reduction ratio: {1 - candidate_count_after/candidate_count_before:.2%}")

## Stage 3: Deterministic Baseline Results

In [ ]:
# Load features for baseline evaluation
features = pd.read_csv(f"{OUTPUT_DIR}/pairwise_features.csv")

# Deterministic baseline: threshold on average similarity
features['baseline_score'] = (features['customer_similarity'] + features['installer_similarity'] + features['address_similarity'] + features['city_match'] + features['state_match']) / 5
features['baseline_pred'] = (features['baseline_score'] >= 0.8).astype(int)

from sklearn.metrics import precision_score, recall_score, f1_score

y_true = features['label']
y_pred = features['baseline_pred']

print("Deterministic Baseline Results:")
print(f"  Precision: {precision_score(y_true, y_pred, zero_division=0):.3f}")
print(f"  Recall: {recall_score(y_true, y_pred, zero_division=0):.3f}")
print(f"  F1: {f1_score(y_true, y_pred, zero_division=0):.3f}")
print(f"\nScore distribution:")
print(features['baseline_score'].describe())

## Baseline on Real Dirty Data (DeepMatcher)

Same deterministic baseline (normalized exact match on title) applied to labeled dirty vs clean Walmart-Amazon pairs. Dirty records have emptied/moved attributes — the baseline should degrade vs clean.

In [ ]:
RAW_DIR = "D:/data viewer/data/raw"
DM_DIR = f"{RAW_DIR}/deepmatcher"

def dm_title_match(row):
    a = normalize_text(row.get("left_title"))
    b = normalize_text(row.get("right_title"))
    return 1 if a and a == b else 0

for name, path in [
    ("dirty_walmart_amazon", f"{DM_DIR}/dirty_walmart_amazon/test.csv"),
    ("clean_walmart_amazon", f"{DM_DIR}/walmart_amazon/test.csv"),
    ("dirty_dblp_acm", f"{DM_DIR}/dirty_dblp_acm/test.csv"),
]:
    df = pd.read_csv(path)
    df["pred"] = df.apply(dm_title_match, axis=1)
    y_true, y_pred = df["label"].values, df["pred"].values
    print(f"{name} ({len(df)} pairs, {int(y_true.sum())} matches):")
    print(f"  missing cells: {int(df.isnull().sum().sum())}")
    print(f"  precision={precision_score(y_true, y_pred, zero_division=0):.3f} "
          f"recall={recall_score(y_true, y_pred, zero_division=0):.3f} "
          f"f1={f1_score(y_true, y_pred, zero_division=0):.3f}")